In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
df = pd.read_csv("../artifacts/Ground_clean_dataset.csv")
df.head()

,Unnamed: 0,Ground Name,City,District,Province,Ground Type,Pitch,Length,Width,Lights,Parking,Washrooms,Pavilion,Scoreboard,Rooms,Parking Cap.,Price (LKR),Rating
0,1.0,Ambalangoda Esplanade,Ambalangoda,Galle,Southern,Public,turf,137.0,149.0,Yes,Yes,Yes,Yes,Yes,2.0,288.0,162417.0,5.0
1,2.0,Ambalangoda High School Ground,Ambalangoda,Galle,Southern,School,matting,100.0,104.0,No,Yes,Yes,Yes,No,2.0,27.0,26760.0,3.0
2,3.0,Ambalangoda Oval,Ambalangoda,Galle,Southern,Public,turf,135.0,134.0,Yes,Yes,Yes,Yes,Yes,2.0,268.0,147946.0,4.0
3,4.0,Ambalangoda Playground,Ambalangoda,Galle,Southern,Public,matting,112.0,105.0,No,Yes,Yes,No,Yes,1.0,26.0,7371.0,2.0
4,5.0,Ambalangoda Public Ground,Ambalangoda,Galle,Southern,Public,matting,121.0,106.0,No,Yes,Yes,No,Yes,1.0,29.0,11911.0,3.0


In [3]:
print(df.columns)

Index(['Unnamed: 0', 'Ground Name', 'City', 'District', 'Province',
       'Ground Type', 'Pitch', 'Length', 'Width', 'Lights', 'Parking',
       'Washrooms', 'Pavilion', 'Scoreboard', 'Rooms', 'Parking Cap.',
       'Price (LKR)', 'Rating'],
      dtype='object')


In [4]:
import pandas as pd

booking_file = "../artifacts/bookings.csv"

bookings = pd.read_csv(booking_file)

bookings.head()

,ground_name,date,booked_by
0,NCC Cricket Ground,2026-04-10,Team Alpha


In [5]:
# 1. IMPORTS
import pandas as pd
import joblib

# 2. LOAD DATASET
df = pd.read_csv("../artifacts/Ground_clean_dataset.csv")

# 3. CLEAN COLUMNS
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# 4. ENCODE DATA 
df = pd.get_dummies(df)

# 5. LOAD MODEL
model = joblib.load("../artifacts/rf_model.joblib")
scaler = joblib.load("../artifacts/scaler.joblib")
model_features = joblib.load("../artifacts/model_features1.joblib")

# 6. LOAD BOOKINGS
bookings = pd.read_csv("../artifacts/bookings.csv")

# 7. LOAD TEAM STATS
team_stats = pd.read_csv("../artifacts/team_stats.csv")

In [6]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

In [7]:
bookings = pd.DataFrame(columns=[
    "ground_name",
    "date",
    "booked_by"
])

In [8]:
def search_grounds(city=None, pitch=None, max_price=None):

    data = df.copy()

    if city:
        data = data[data["city"].str.lower() == city.lower()]

    if pitch:
        data = data[data["pitch"].str.lower() == pitch.lower()]

    if max_price:
        data = data[data["price_(lkr)"] <= max_price]

    return data[[
        "ground_name",
        "city",
        "pitch",
        "price_(lkr)"
    ]]

In [9]:
def check_availability(ground_name, date):

    conflict = bookings[
        (bookings["ground_name"] == ground_name) &
        (bookings["date"] == date)
    ]

    if conflict.empty:
        print(f"{ground_name} on {date} → Available")
    else:
        print(f"{ground_name} on {date} → Already Booked")

In [10]:
def remove_booked_grounds(data, date):

    booked = bookings[bookings["date"] == date]["ground_name"]

    data = data[~data["ground_name"].isin(booked)]

    return data

In [11]:
# Convert categorical columns to one-hot encoding
data_encoded = pd.get_dummies(df)

# Ensure all training features exist
data_encoded = data_encoded.reindex(columns=model_features, fill_value=0)

data_encoded = data_encoded[model_features]

In [12]:
import joblib

model = joblib.load("../artifacts/rf_model.joblib")
scaler = joblib.load("../artifacts/scaler.joblib")
model_features = joblib.load("../artifacts/model_features1.joblib")

def recommend_grounds(user_features, date,
                      province=None,
                      district=None,
                      city=None,
                      pitch=None,
                      max_price=None,
                      top_n=5):

    data = df.copy()

    # Apply location filters
    if province:
        data = data[data["province"].str.lower() == province.lower()]

    if district:
        data = data[data["district"].str.lower() == district.lower()]

    if city:
        data = data[data["city"].str.lower() == city.lower()]

    if pitch:
        data = data[data["pitch"].str.lower() == pitch.lower()]

    if max_price:
        data = data[data["price_(lkr)"] <= max_price]

    # Remove booked grounds
    if not bookings.empty:
        booked = bookings[bookings["date"] == date]["ground_name"]
        data = data[~data["ground_name"].isin(booked)]

    scores = []

    for _, row in data.iterrows():

        features = {
            "price_(lkr)": row.get("price_(lkr)", 0),
            "lights": row.get("lights", 0),
            "parking": row.get("parking", 0),
            "rating": row.get("rating", 0)
        }

        # Apply user preferences
        features.update(user_features)

        input_df = pd.DataFrame([features])
        input_df = input_df.reindex(columns=model_features, fill_value=0)

        input_scaled = scaler.transform(input_df)

        score = model.predict_proba(input_scaled)[0][1]

        scores.append(score)

    data["score"] = scores

    return data.sort_values(by="score", ascending=False).head(top_n)

In [13]:
import os

booking_file = "../artifacts/bookings.csv"

if os.path.exists(booking_file):
    bookings = pd.read_csv(booking_file)
else:
    bookings = pd.DataFrame(columns=["ground_name","date","booked_by"])

In [14]:
def book_ground(ground_name, date, booked_by):

    global bookings

    conflict = bookings[
        (bookings["ground_name"] == ground_name) &
        (bookings["date"] == date)
    ]

    if not conflict.empty:
        print(" Already booked")
        return

    new_booking = {
        "ground_name": ground_name,
        "date": date,
        "booked_by": booked_by
    }

    bookings = pd.concat([bookings, pd.DataFrame([new_booking])], ignore_index=True)

    bookings.to_csv("../artifacts/bookings.csv", index=False)

    print(" Booking Confirmed")

In [15]:
df.columns

Index(['unnamed:_0', 'length', 'width', 'rooms', 'parking_cap.', 'price_(lkr)',
       'rating', 'ground_name_air_force_ground_katunayake',
       'ground_name_ambalangoda_beachside_ground',
       'ground_name_ambalangoda_central_field',
       ...
       'pitch_matting', 'pitch_turf', 'lights_no', 'lights_yes', 'parking_yes',
       'washrooms_yes', 'pavilion_no', 'pavilion_yes', 'scoreboard_no',
       'scoreboard_yes'],
      dtype='object', length=1050)

In [16]:
search_grounds(
    city="Colombo",
    pitch="turf",
    max_price=150000
)

KeyError: 'city'

In [ ]:
user_input = {
    "price_(lkr)":120000,
    "lights":1,
    "parking":1,
    "rating":4
}

In [ ]:
v = recommend_grounds(
    user_input,
    date="2026-04-10"
)

In [ ]:
v

In [ ]:
book_ground("NCC Cricket Ground","2026-04-10","Team Alpha")
recommend_grounds(user_input,"2026-04-10")

In [ ]:
book_ground("NCC Cricket Ground","2026-04-10","Team Alpha")

In [ ]:
user_input = {
    "price_(lkr)": 120000,
    "lights": 1,
    "parking": 1,
    "rating": 4
}

results = recommend_grounds(
    user_input,
    date="2026-04-10",
    province="Western",
    district="Colombo",
    city="Colombo",
    pitch="turf",
    max_price=150000
)

results[["ground_name","city","pitch","price_(lkr)","score"]]

In [18]:
df

,unnamed:_0,length,width,rooms,parking_cap.,price_(lkr),rating,ground_name_air_force_ground_katunayake,ground_name_ambalangoda_beachside_ground,ground_name_ambalangoda_central_field,...,pitch_matting,pitch_turf,lights_no,lights_yes,parking_yes,washrooms_yes,pavilion_no,pavilion_yes,scoreboard_no,scoreboard_yes
0,1.0,137.0,149.0,2.0,288.0,162417.0,5.0,False,False,False,...,False,True,False,True,True,True,False,True,False,True
1,2.0,100.0,104.0,2.0,27.0,26760.0,3.0,False,False,False,...,True,False,True,False,True,True,False,True,True,False
2,3.0,135.0,134.0,2.0,268.0,147946.0,4.0,False,False,False,...,False,True,False,True,True,True,False,True,False,True
3,4.0,112.0,105.0,1.0,26.0,7371.0,2.0,False,False,False,...,True,False,True,False,True,True,True,False,False,True
4,5.0,121.0,106.0,1.0,29.0,11911.0,3.0,False,False,False,...,True,False,True,False,True,True,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1033,1032.0,112.0,109.0,1.0,13.0,7572.0,2.0,False,False,False,...,True,False,True,False,True,True,True,False,False,True
1034,1033.0,112.0,109.0,1.0,13.0,7572.0,2.0,False,False,False,...,True,False,True,False,True,True,True,False,False,True
1035,1034.0,112.0,109.0,1.0,13.0,7572.0,2.0,False,False,False,...,True,False,True,False,True,True,True,False,False,True
1036,1035.0,112.0,109.0,1.0,13.0,7572.0,2.0,False,False,False,...,True,False,True,False,True,True,True,False,False,True


In [19]:
bookings_df = pd.read_csv("artifacts/bookings.csv")
bookings_df

,ground_name,date,booked_by
0,NCC Cricket Ground,2026-03-25,colombo cc
1,Police Park Ground,2026-10-30,colombo cc
